In [19]:
# Instalamos python-dotenv para poder cargar la API key de forma segura desde el archivo .env
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [20]:
#KARENS_API_KEY

# Importamos las herramientas necesarias para cargar y leer las variables del archivo .env
from dotenv import load_dotenv
import os

# Cargamos las variables guardadas en el archivo .env
load_dotenv()

# Guardamos la API key en una variable para poder utilizarla más adelante
api_key = os.getenv("KARENS_API_KEY")

In [21]:
# Comprobamos que la API key se ha cargado correctamente sin mostrar su contenido
if api_key:
    print("API key cargada correctamente")
else:
    print("No se ha podido cargar la API key")

API key cargada correctamente


In [22]:
# Importamos requests para poder realizar peticiones HTTP a la API
import requests

# Definimos el endpoint de The Cat API que utilizaremos para consultar las razas
url = "https://api.thecatapi.com/v1/breeds"

In [23]:
# Creamos los headers para enviar nuestra API key de forma segura en la petición
headers = {
    "x-api-key": api_key
}

# Realizamos una petición GET al endpoint de razas
response = requests.get(url, headers=headers)

# Comprobamos el código de estado de la respuesta
print(response.status_code)

200


In [24]:
# Convertimos la respuesta de la API de formato JSON a un objeto de Python
data = response.json()

# Comprobamos qué tipo de objeto hemos recibido
print(type(data))

# Comprobamos cuántos registros hemos obtenido
print(len(data))

<class 'list'>
107


In [25]:
import pandas as pd

In [26]:
# Convertimos los datos obtenidos de la API en un DataFrame
df_cats = pd.DataFrame(data)

# Comprobamos las dimensiones del DataFrame
print(df_cats.shape)

(107, 18)


In [27]:
# Comprobamos la estructura del DataFrame y los tipos de datos de cada columna
df_cats.info()

<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  107 non-null    str   
 1   name                107 non-null    str   
 2   species_id          107 non-null    str   
 3   life_span           107 non-null    str   
 4   temperament         107 non-null    str   
 5   origin              100 non-null    str   
 6   country_codes       100 non-null    str   
 7   country_code        100 non-null    str   
 8   description         107 non-null    str   
 9   bred_for            0 non-null      object
 10  perfect_for         0 non-null      object
 11  breed_group         100 non-null    str   
 12  history             100 non-null    str   
 13  alt_names           55 non-null     str   
 14  reference_image_id  66 non-null     str   
 15  weight              107 non-null    object
 16  height              107 non-null    o

In [28]:
# Seleccionamos las columnas que nos interesan para el análisis
columns_analysis = [
    "name",
    "life_span",
    "temperament",
    "origin",
    "country_codes",
    "country_code",
    "description",
    "breed_group",
    "weight",
    "height"
]

# Creamos un nuevo DataFrame solo con las variables seleccionadas
df_cat_analysis = df_cats[columns_analysis].copy()

# Mostramos las primeras filas
df_cat_analysis.head()

,name,life_span,temperament,origin,country_codes,country_code,description,breed_group,weight,height
0,Abyssinian,14-17,"Active, Energetic, Independent, Intelligent, G...",Egypt,EG,EG,"Medium-sized, elegant cat with a distinctive t...",Short-haired,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}"
1,Aegean,9-12,"Affectionate, Social, Intelligent, Playful, Ac...",Greece,GR,GR,Medium-sized natural breed with a semi-longhai...,Semi-longhair,"{'imperial': '9-12', 'metric': '4-5.4'}","{'imperial': '9-11', 'metric': '23-28'}"
2,American Bobtail,11-15,"Intelligent, Interactive, Lively, Playful, Sen...",United States,US,US,Medium to large cat with a distinctive natural...,Short/Medium-hair,"{'imperial': '12-16', 'metric': '5.4-7.3'}","{'imperial': '11-14', 'metric': '28-36'}"
3,American Curl,12-16,"Affectionate, Curious, Intelligent, Interactiv...",United States,US,US,Medium-sized cat with distinctive backward-cur...,Short/Long-hair,"{'imperial': '7-11', 'metric': '3.2-5'}","{'imperial': '9-12', 'metric': '23-30'}"
4,American Ringtail,12-18,"Friendly, Affectionate, Intelligent, Active, P...",United States,US,US,Medium-sized cat with distinctive curled tail ...,Short/Long-hair,"{'imperial': '7-15', 'metric': '3.2-6.8'}","{'imperial': '9-12', 'metric': '23-30'}"


In [29]:
# Mostramos los diferentes valores que contiene la columna life_span
df_cat_analysis["life_span"].unique()

<StringArray>
['14-17',  '9-12', '11-15', '12-16', '12-18', '15-20', '14-18', '12-15',
 '12-14', '12-20', '14-20', '12-17', '15-18', '10-15', '10-14',     '1',
 '14-16',  '8-14',     '5', '13-16',  '9-15', '10-12',     '4', '11-16',
 '11-14', '14-15', '13-15', '11-18', '14-19']
Length: 29, dtype: str

In [30]:
# Buscamos registros de life_span que no contienen un rango
df_cat_analysis[
    ~df_cat_analysis["life_span"].str.contains("-")
][["name", "life_span"]]

,name,life_span
24,Caracat,1
40,Dwelf,5
42,Elf Cat,5
56,Kellas Cat,1
68,Napoleon Longhair,4
75,Pantherette,1
92,Singapura Tabby,1


In [31]:
# Creamos el filtro para los registros cuyo life_span no contiene un rango
life_span_single = ~df_cat_analysis["life_span"].str.contains("-")

# Mostramos información adicional de estos registros
df_cat_analysis.loc[
    life_span_single,
    [
        "name",
        "life_span",
        "origin",
        "description",

    ]
]

,name,life_span,origin,description
24,Caracat,1,NaN,Large hybrid between caracal and domestic cat....
40,Dwelf,5,NaN,"Small, hairless cat with curled ears and short..."
42,Elf Cat,5,NaN,Medium-sized hairless cat with distinctive cur...
56,Kellas Cat,1,NaN,"Large, melanistic wild-domestic hybrid found i..."
68,Napoleon Longhair,4,NaN,Small to medium cat with short legs and long p...
75,Pantherette,1,NaN,"Medium to large, muscular black cat bred to re..."
92,Singapura Tabby,1,NaN,"Small, muscular cat with tabby pattern variati..."


In [32]:
# Mostramos el contenido completo de las columnas de texto
pd.set_option("display.max_colwidth", None)

In [33]:
# Restauramos la visualización por defecto
pd.reset_option("display.max_colwidth")

In [34]:
# Creamos el filtro para los registros cuyo life_span contiene un rango
life_span_valid = df_cat_analysis["life_span"].str.contains("-")

# Mostramos información adicional de estos registros
df_cat_analysis.loc[
    life_span_valid,
    [
    "name",
    "life_span",
    "temperament",
    "origin",
    "country_codes",
    "country_code",
    "description",
    "breed_group",
    "weight",
    "height"
    ]
]

,name,life_span,temperament,origin,country_codes,country_code,description,breed_group,weight,height
0,Abyssinian,14-17,"Active, Energetic, Independent, Intelligent, G...",Egypt,EG,EG,"Medium-sized, elegant cat with a distinctive t...",Short-haired,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}"
1,Aegean,9-12,"Affectionate, Social, Intelligent, Playful, Ac...",Greece,GR,GR,Medium-sized natural breed with a semi-longhai...,Semi-longhair,"{'imperial': '9-12', 'metric': '4-5.4'}","{'imperial': '9-11', 'metric': '23-28'}"
2,American Bobtail,11-15,"Intelligent, Interactive, Lively, Playful, Sen...",United States,US,US,Medium to large cat with a distinctive natural...,Short/Medium-hair,"{'imperial': '12-16', 'metric': '5.4-7.3'}","{'imperial': '11-14', 'metric': '28-36'}"
3,American Curl,12-16,"Affectionate, Curious, Intelligent, Interactiv...",United States,US,US,Medium-sized cat with distinctive backward-cur...,Short/Long-hair,"{'imperial': '7-11', 'metric': '3.2-5'}","{'imperial': '9-12', 'metric': '23-30'}"
4,American Ringtail,12-18,"Friendly, Affectionate, Intelligent, Active, P...",United States,US,US,Medium-sized cat with distinctive curled tail ...,Short/Long-hair,"{'imperial': '7-15', 'metric': '3.2-6.8'}","{'imperial': '9-12', 'metric': '23-30'}"
...,...,...,...,...,...,...,...,...,...,...
102,Toyger,12-15,"Playful, Social, Intelligent, Friendly, Outgoi...",United States,US,US,"Medium to large, muscular cat with distinctive...",Short-haired,"{'imperial': '10-15', 'metric': '4.5-6.8'}","{'imperial': '11-14', 'metric': '28-36'}"
103,Turkish Angora,12-18,"Affectionate, Agile, Clever, Gentle, Intellige...",Turkey,TR,TR,"Medium-sized, elegant cat with long silky coat...",Long-haired,"{'imperial': '7-14', 'metric': '3.2-6.4'}","{'imperial': '10-14', 'metric': '25-36'}"
104,Turkish Van,12-17,"Agile, Intelligent, Loyal, Playful, Energetic,...",Turkey,TR,TR,"Large, muscular cat with distinctive Van patte...",Long-haired,"{'imperial': '10-20', 'metric': '4.5-9.1'}","{'imperial': '13-17', 'metric': '33-43'}"
105,Ukrainian Levkoy,12-15,"Affectionate, Relaxed, Playful, Curious, Intel...",Ukraine,UA,UA,"Medium-sized, hairless cat with distinctive fo...",Hairless,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}"


Se detectaron 7 registros con anomalías estructurales. Se compararon con los registros correctamente formateados para determinar si el problema afectaba únicamente a variables concretas o al registro completo.

In [38]:
#Vamos a crear life_span_min y life_span_max a partir de la columna life_span. Para ello, primero vamos a separar los valores de life_span en dos columnas nuevas.
# Separamos life_span en dos columnas usando el guion
life_span_split = (
    df_cat_analysis.loc[life_span_valid, "life_span"]
    .str.split("-", expand=True)
)

life_span_split.head()


,0,1
0,14,17
1,9,12
2,11,15
3,12,16
4,12,18


In [39]:
# Añadimos las columnas de esperanza de vida mínima y máxima
# y convertimos sus valores a tipo numérico
df_cat_analysis.loc[life_span_valid, "life_span_min"] = (
    pd.to_numeric(life_span_split[0])
)

df_cat_analysis.loc[life_span_valid, "life_span_max"] = (
    pd.to_numeric(life_span_split[1])
)

In [42]:
# Comprobamos las nuevas columnas
df_cat_analysis[
    ["name", "life_span", "life_span_min", "life_span_max"]
].head()

,name,life_span,life_span_min,life_span_max
0,Abyssinian,14-17,14.0,17.0
1,Aegean,9-12,9.0,12.0
2,American Bobtail,11-15,11.0,15.0
3,American Curl,12-16,12.0,16.0
4,American Ringtail,12-18,12.0,18.0


In [45]:
df_cat_analysis.head()

,name,life_span,temperament,origin,country_codes,country_code,description,breed_group,weight,height,life_span_min,life_span_max
0,Abyssinian,14-17,"Active, Energetic, Independent, Intelligent, G...",Egypt,EG,EG,"Medium-sized, elegant cat with a distinctive t...",Short-haired,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}",14.0,17.0
1,Aegean,9-12,"Affectionate, Social, Intelligent, Playful, Ac...",Greece,GR,GR,Medium-sized natural breed with a semi-longhai...,Semi-longhair,"{'imperial': '9-12', 'metric': '4-5.4'}","{'imperial': '9-11', 'metric': '23-28'}",9.0,12.0
2,American Bobtail,11-15,"Intelligent, Interactive, Lively, Playful, Sen...",United States,US,US,Medium to large cat with a distinctive natural...,Short/Medium-hair,"{'imperial': '12-16', 'metric': '5.4-7.3'}","{'imperial': '11-14', 'metric': '28-36'}",11.0,15.0
3,American Curl,12-16,"Affectionate, Curious, Intelligent, Interactiv...",United States,US,US,Medium-sized cat with distinctive backward-cur...,Short/Long-hair,"{'imperial': '7-11', 'metric': '3.2-5'}","{'imperial': '9-12', 'metric': '23-30'}",12.0,16.0
4,American Ringtail,12-18,"Friendly, Affectionate, Intelligent, Active, P...",United States,US,US,Medium-sized cat with distinctive curled tail ...,Short/Long-hair,"{'imperial': '7-15', 'metric': '3.2-6.8'}","{'imperial': '9-12', 'metric': '23-30'}",12.0,18.0


axis=0 → calcula por columnas.
axis=1 → calcula por filas. ← Esto queremos.

In [46]:
# Calculamos la esperanza de vida media de cada raza
df_cat_analysis["life_span_mean"] = df_cat_analysis[
    ["life_span_min", "life_span_max"]
].mean(axis=1)

In [47]:
# Comprobamos las nuevas columnas
df_cat_analysis[
    ["name", "life_span", "life_span_min", "life_span_max", "life_span_mean"]
].head()

,name,life_span,life_span_min,life_span_max,life_span_mean
0,Abyssinian,14-17,14.0,17.0,15.5
1,Aegean,9-12,9.0,12.0,10.5
2,American Bobtail,11-15,11.0,15.0,13.0
3,American Curl,12-16,12.0,16.0,14.0
4,American Ringtail,12-18,12.0,18.0,15.0


In [48]:
# Mostramos los diferentes valores que contiene la columna life_span
df_cat_analysis["life_span_mean"].unique()

array([15.5, 10.5, 13. , 14. , 15. , 17.5, 16. , 13.5, 17. , 14.5, 16.5,
       12.5, 12. ,  nan, 11. ])

In [49]:
#nan corresponde anomalias, eliminamos los registros del dataframe.
# Creamos un DataFrame sin los registros anómalos de life_span
df_cat_clean = df_cat_analysis.loc[life_span_valid].copy()

# Comprobamos las dimensiones
print(df_cat_clean.shape)

(100, 13)


In [52]:
# Separamos las características de temperament por comas
temperaments = (
    df_cat_clean["temperament"]
    .str.split(",")
    .explode()
    .str.strip()
)


# Mostramos los diferentes temperamentos
temperaments.unique()

<StringArray>
[                                       'Active',
                                     'Energetic',
                                   'Independent',
                                   'Intelligent',
                                        'Gentle',
                                       'Curious',
                                       'Playful',
                                  'Affectionate',
                                        'Social',
                                     'Adaptable',
                                   'Interactive',
                                        'Lively',
                                     'Sensitive',
                                       'Devoted',
                                      'Friendly',
                                    'Easy Going',
                                          'Calm',
                                         'Loyal',
                                      'Sensible',
                                    

In [53]:
# Contamos cuántas veces aparece cada temperamento
temperament_counts = temperaments.value_counts()

temperament_counts

temperament
Affectionate            74
Intelligent             69
Playful                 69
Social                  54
Active                  44
                        ..
Easygoing                1
Docile                   1
Silly                    1
Easily leash-trained     1
Tenacious                1
Name: count, Length: 63, dtype: int64

Esto ya nos da una pista bastante potente: en las 100 razas válidas, los temperamentos dominantes aparecen muchísimas veces

In [54]:
# Mostramos todos los temperamentos y su frecuencia
for temperament, count in temperament_counts.items():
    print(f"{temperament}: {count}")

Affectionate: 74
Intelligent: 69
Playful: 69
Social: 54
Active: 44
Gentle: 35
Loyal: 34
Curious: 27
Friendly: 21
Lively: 14
Calm: 14
Sweet: 14
Independent: 12
Quiet: 11
Easy Going: 10
Agile: 10
Interactive: 9
Energetic: 7
Adaptable: 7
Athletic: 7
Loving: 7
Devoted: 6
Dependent: 5
Demanding: 4
Patient: 4
Clever: 4
Vocal: 4
Outgoing: 4
Relaxed: 3
Mischievous: 3
Varies widely - typically Friendly: 3
Highly intelligent: 3
Sensitive: 2
Alert: 2
Peaceful: 2
Confident: 2
Trainable: 2
Inquisitive: 2
Sociable: 2
People-oriented: 2
Sweet-tempered: 2
Reserved: 2
Adventurous: 2
Sensible: 1
Fun-loving: 1
Tolerant: 1
Observant: 1
Highly interactive: 1
Talkative: 1
Warm: 1
Expressive: 1
Personable: 1
Cheerful: 1
Charming: 1
Extremely affectionate: 1
Shy: 1
Sedate: 1
Varies by individual - typically Affectionate: 1
Easygoing: 1
Docile: 1
Silly: 1
Easily leash-trained: 1
Tenacious: 1


In [55]:
# Unificamos variaciones claras de una misma característica
temperament_replace = {
    "Easygoing": "Easy Going",
    "Highly intelligent": "Intelligent",
    "Highly interactive": "Interactive",
    "Extremely affectionate": "Affectionate",
    "Sociable": "Social"
}

In [56]:
# Probamos la unificación sin modificar el DataFrame
temperaments_clean = temperaments.replace(temperament_replace)

print("Temperamentos antes:", temperaments.nunique())
print("Temperamentos después:", temperaments_clean.nunique())

Temperamentos antes: 63
Temperamentos después: 58


In [57]:
# Contamos de nuevo la frecuencia después de unificar los temperamentos
temperament_counts_clean = temperaments_clean.value_counts()

for temperament, count in temperament_counts_clean.items():
    print(f"{temperament}: {count}")

Affectionate: 75
Intelligent: 72
Playful: 69
Social: 56
Active: 44
Gentle: 35
Loyal: 34
Curious: 27
Friendly: 21
Lively: 14
Calm: 14
Sweet: 14
Independent: 12
Easy Going: 11
Quiet: 11
Interactive: 10
Agile: 10
Energetic: 7
Adaptable: 7
Athletic: 7
Loving: 7
Devoted: 6
Dependent: 5
Demanding: 4
Patient: 4
Clever: 4
Vocal: 4
Outgoing: 4
Relaxed: 3
Mischievous: 3
Varies widely - typically Friendly: 3
Sensitive: 2
Alert: 2
Peaceful: 2
Confident: 2
Trainable: 2
Inquisitive: 2
People-oriented: 2
Sweet-tempered: 2
Reserved: 2
Adventurous: 2
Sensible: 1
Fun-loving: 1
Tolerant: 1
Observant: 1
Talkative: 1
Warm: 1
Expressive: 1
Personable: 1
Cheerful: 1
Charming: 1
Shy: 1
Sedate: 1
Varies by individual - typically Affectionate: 1
Docile: 1
Silly: 1
Easily leash-trained: 1
Tenacious: 1


In [58]:
# Añadimos las etiquetas que indican el temperamento típico
temperament_replace.update({
    "Varies widely - typically Friendly": "Friendly",
    "Varies by individual - typically Affectionate": "Affectionate"
})

# Volvemos a aplicar la normalización
temperaments_clean = temperaments.replace(temperament_replace)

print("Temperamentos diferentes:", temperaments_clean.nunique())

Temperamentos diferentes: 56


In [59]:
# Mostramos todos los temperamentos distintos ya normalizados
for temperament in sorted(temperaments_clean.unique()):
    print(temperament)

Active
Adaptable
Adventurous
Affectionate
Agile
Alert
Athletic
Calm
Charming
Cheerful
Clever
Confident
Curious
Demanding
Dependent
Devoted
Docile
Easily leash-trained
Easy Going
Energetic
Expressive
Friendly
Fun-loving
Gentle
Independent
Inquisitive
Intelligent
Interactive
Lively
Loving
Loyal
Mischievous
Observant
Outgoing
Patient
Peaceful
People-oriented
Personable
Playful
Quiet
Relaxed
Reserved
Sedate
Sensible
Sensitive
Shy
Silly
Social
Sweet
Sweet-tempered
Talkative
Tenacious
Tolerant
Trainable
Vocal
Warm
